In [1]:
import os
import sys
import time
from pathlib import Path
from dotenv import load_dotenv

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

# Load environment
load_dotenv(project_root / ".env")

# Check for API key (OpenRouter preferred, OpenAI as fallback)
openrouter_key = os.getenv("OPENROUTER_API_KEY")
openai_key = os.getenv("OPENAI_API_KEY")

if not openrouter_key and not openai_key:
    raise EnvironmentError(
        "❌ No API key found!\n"
        "   Add OPENROUTER_API_KEY (recommended) or OPENAI_API_KEY to .env"
    )

# Load configuration
from real_state.config import (
    VECTOR_DIR, CACHE_DIR, TOP_K_RESULTS,
    CHAT_MODEL, EMBEDDING_MODEL, PROVIDER
)

provider = "OpenRouter" if openrouter_key else "OpenAI"
print("✅ Environment loaded")
print(f"🌐 Provider: {provider}")
print(f"📁 Project root: {project_root}")

✅ Environment loaded
🌐 Provider: OpenRouter
📁 Project root: d:\ai\bootcamp\mini-project-02\real_state


## Import chat services
Using RAG/CAG/CRAG services from application layer (NOT defined here!)

In [2]:
from real_state.application.chat_service import (
    RagService,
    CAGService,
    CRAGService,
    CAGCache
)

print("Chat services loaded from sevice layer")
print("📍 Location: context_engineering.application.chat_service")
print("\n🎯 Available services:")
print("   1. RAGService   - Standard RAG with modern LCEL")
print("   2. CAGService   - Cache-Augmented Generation (semantic)")
print("   3. CRAGService  - Corrective RAG with confidence scoring")
print("   4. CAGCache     - Semantic cache (FAQs + History)")

Chat services loaded from sevice layer
📍 Location: context_engineering.application.chat_service

🎯 Available services:
   1. RAGService   - Standard RAG with modern LCEL
   2. CAGService   - Cache-Augmented Generation (semantic)
   3. CRAGService  - Corrective RAG with confidence scoring
   4. CAGCache     - Semantic cache (FAQs + History)


In [3]:
from real_state.infrastructure.llm_provider import (
    get_default_embeddings,
    get_chat_llm

)
from real_state.infrastructure.db.vector_db import QdrantStorage


emb = get_default_embeddings()
llm = get_chat_llm(temperature=0)

print(f"✅ LLM initialized: {CHAT_MODEL}")
print(f"✅ Embeddings initialized: {EMBEDDING_MODEL}")
print(f"🌐 Provider: {PROVIDER}")

qdrant_client = QdrantStorage()

try:
    qdrant_client.collection_info(collection_name="prime_lands")
except:
    print("❌ Collection not found")




✅ LLM initialized: google/gemini-2.5-flash
✅ Embeddings initialized: openai/text-embedding-3-large
🌐 Provider: openrouter


---
# 1️⃣ Standard RAG with Service

In [4]:
rag_service = RagService(
    retriever = qdrant_client,
    llm = llm,
    k = TOP_K_RESULTS
)

print("✅ RAGService initialized")
print(f"🎯 Retrieval: top-{TOP_K_RESULTS} documents")

✅ RAGService initialized
🎯 Retrieval: top-4 documents


In [5]:
USER_QUERY = "how much SKYE BLOSSOM - KOTTAWA?"
print(f"🔍 Query: {USER_QUERY}\n")
print("=" * 80)
print("GENERATING ANSWER WITH RAG SERVICE...")
print("=" * 80)

result = rag_service.generate(USER_QUERY)

print(f"\n⏱️  Generation time: {result['generation_time']:.2f}s")
print(f"📄 Documents used: {result['num_docs']}")
print("\n" + "=" * 80)
print("ANSWER")
print("=" * 80)
print(result['answer'])
print("\n" + "=" * 80)
print("EVIDENCE URLS")
print("=" * 80)
for url in result['evidence_urls']:
    print(f"  - {url}")

🔍 Query: how much SKYE BLOSSOM - KOTTAWA?

GENERATING ANSWER WITH RAG SERVICE...

⏱️  Generation time: 7.73s
📄 Documents used: 4

ANSWER
**Key Facts**
*   Skye Blossom Kottawa is an apartment project located in Kottawa. [https://www.primelands.lk/apartment/SKYE-BLOSSOM-KOTTAWA/en]
*   It is described as the only high-rise of its kind in the thriving suburb of Kottawa. [https://www.primelands.lk/apartment/SKYE-BLOSSOM-KOTTAWA/en]
*   The project is identified as a rare investment opportunity with strong value appreciation potential. [https://www.primelands.lk/apartment/SKYE-BLOSSOM-KOTTAWA/en]
*   The area experiences exceptionally high rental demand. [https://www.primelands.lk/apartment/SKYE-BLOSSOM-KOTTAWA/en]

**Answer**
The price for Skye Blossom Kottawa is 50,000,000 LKR [https://www.primelands.lk/apartment/SKYE-BLOSSOM-KOTTAWA/en]. This project is located in Kottawa and is considered a prestigious address with high rental demand [https://www.primelands.lk/apartment/SKYE-BLOSSOM-KO

---
# 2️⃣ Cache-Augmented Generation (CAG) with Semantic Matching

In [5]:
CACHE_DIR.mkdir(parents=True, exist_ok=True)
cache = CAGCache(
    cache_dir=CACHE_DIR,
    embedder=emb, 
    similarity_threshold=0.90,  
    history_ttl_hours=24  
)

cag_service = CAGService(
    rag_service = rag_service,
    cache=cache
)

print("✅ CAGService initialized (semantic matching)")
print(f"📁 Cache directory: {CACHE_DIR}")
stats = cache.stats()
print(f"📊 Cached responses: {stats['total_cached']}")
print(f"🎯 Similarity threshold: {stats['similarity_threshold']}")
print(f"⏰ History TTL: {stats['history_ttl_hours']}h")

✅ CAGService initialized (semantic matching)
📁 Cache directory: d:\ai\bootcamp\mini-project-02\real_state\data\cag_cache
📊 Cached responses: 22
🎯 Similarity threshold: 0.9
⏰ History TTL: 24h


In [6]:
from real_state.config import KNOWN_FAQS

print(f"📋 Found {len(KNOWN_FAQS)} FAQs in config/faqs.yaml")
print("\n📝 Sample FAQs:")
for faq in KNOWN_FAQS[:5]:
    print(f"   - {faq}")
print(f"   ... and {len(KNOWN_FAQS) - 5} more\n")

# Load FAQs into cache (this just registers them, doesn't generate responses yet)
loaded = cag_service.load_faqs(KNOWN_FAQS)
print(f"✅ Loaded {loaded} new FAQs into cache")

# To warm FAQs (generate responses), uncomment:
# print("\n🔥 Warming FAQs (this may take a few minutes)...")
# cag_service.warm_faqs()

print("\n💡 Tip: Run cag_service.warm_faqs() to pre-generate FAQ responses")

📋 Found 22 FAQs in config/faqs.yaml

📝 Sample FAQs:
   - What types of properties does Prime Group offer?
   - How much does SKYE BLOSSOM KOTTAWA cost?
   - What are the available payment plans at PrimeLands?
   - Which locations does PrimeLands have projects in?
   - What amenities are included in PrimeLands housing projects?
   ... and 17 more

✅ Loaded 0 new FAQs into cache

💡 Tip: Run cag_service.warm_faqs() to pre-generate FAQ responses


In [7]:
from real_state.config import CRAG_EXPANDED_K

crag_service = CRAGService(
    retriever=qdrant_client,
    llm = llm,
    initial_k=TOP_K_RESULTS,
    expanded_k=CRAG_EXPANDED_K
)

print("✅ CRAGService initialized")
print(f"🎯 Initial retrieval: top-{TOP_K_RESULTS}")
print(f"🔄 Corrective retrieval: top-{CRAG_EXPANDED_K}")

✅ CRAGService initialized
🎯 Initial retrieval: top-4
🔄 Corrective retrieval: top-8


In [8]:
if 'rag_service' not in dir():
    rag_service = RAGService(retriever=retriever, llm=llm, k=TOP_K_RESULTS)
if 'cag_service' not in dir():
    cache = CAGCache(cache_dir=CACHE_DIR, embedder=embeddings, similarity_threshold=0.90)
    cag_service = CAGService(rag_service=rag_service, cache=cache)
if 'crag_service' not in dir():
    from context_engineering.config import CRAG_EXPANDED_K
    crag_service = CRAGService(retriever=retriever, llm=llm, initial_k=TOP_K_RESULTS, expanded_k=CRAG_EXPANDED_K)

print("=" * 80)
print("🎤 INTERACTIVE RAG INFERENCE")
print("=" * 80)
print("\n📝 Ask your question about Nawaloka Hospital...\n")


#Which PrimeLands projects are close to highway entrances?
YOUR_QUESTION = input("❓ Your question: ")

print(f"\n🔍 Processing query: '{YOUR_QUESTION}'\n")
print("=" * 80)

results = {}


# 1. Standard RAG
print("\n1️⃣  Standard RAG")
print("-" * 80)
start = time.time()
rag_result = rag_service.generate(YOUR_QUESTION)
results['RAG'] = {
    'answer': rag_result.get('answer', 'N/A'),
    'time': rag_result.get('generation_time', rag_result.get('time', 0)),
    'docs': rag_result.get('num_docs', len(rag_result.get('evidence_urls', []))),
    'urls': rag_result.get('evidence_urls', [])
}
print(f"✅ Completed in {results['RAG']['time']:.2f}s")
print(f"📄 Documents retrieved: {results['RAG']['docs']}")
print(f"\n💬 Answer:")
print(results['RAG']['answer'])
print(f"\n📎 Evidence URLs:")
for url in results['RAG']['urls'][:3]:
    print(f"   • {url}")

# 2. Cache-Augmented Generation
print("\n" + "=" * 80)
print("\n2️⃣  Cache-Augmented Generation (CAG)")
print("-" * 80)
cag_result = cag_service.generate(YOUR_QUESTION, use_cache=True, verbose=False)
results['CAG'] = {
    'answer': cag_result.get('answer', 'N/A'),
    'time': cag_result.get('generation_time', cag_result.get('time', 0)),
    'docs': cag_result.get('num_docs', cag_result.get('docs_used', len(cag_result.get('evidence_urls', [])))),
    'cache_hit': cag_result.get('cache_hit', False),
    'urls': cag_result.get('evidence_urls', [])
}
print(f"✅ Completed in {results['CAG']['time']:.2f}s")
print(f"💾 Cache hit: {results['CAG']['cache_hit']}")
print(f"📄 Documents retrieved: {results['CAG']['docs']}")
print(f"\n💬 Answer:")
print(results['CAG']['answer'])
print(f"\n📎 Evidence URLs:")
for url in results['CAG']['urls'][:3]:
    print(f"   • {url}")

# 3. Corrective RAG
print("\n" + "=" * 80)
print("\n3️⃣  Corrective RAG (CRAG)")
print("-" * 80)
crag_result = crag_service.generate(YOUR_QUESTION, confidence_threshold=0.6, verbose=False)
results['CRAG'] = {
    'answer': crag_result.get('answer', 'N/A'),
    'time': crag_result.get('generation_time', crag_result.get('time', 0)),
    'docs': crag_result.get('docs_used', crag_result.get('num_docs', len(crag_result.get('evidence_urls', [])))),
    'confidence': crag_result.get('confidence_final', crag_result.get('confidence', 0.0)),
    'corrected': crag_result.get('correction_applied', False),
    'urls': crag_result.get('evidence_urls', [])
}
print(f"✅ Completed in {results['CRAG']['time']:.2f}s")
print(f"📊 Confidence: {results['CRAG']['confidence']:.2f}")
print(f"🔧 Correction applied: {results['CRAG']['corrected']}")
print(f"📄 Documents used: {results['CRAG']['docs']}")
print(f"\n💬 Answer:")
print(results['CRAG']['answer'])
print(f"\n📎 Evidence URLs:")
for url in results['CRAG']['urls'][:3]:
    print(f"   • {url}")

print("\n" + "=" * 80)
print("📊 PERFORMANCE COMPARISON")
print("=" * 80)
print(f"\n{'System':<15} {'Time (s)':<12} {'Docs':<8} {'Special Feature':<30}")
print("-" * 80)
print(f"{'Standard RAG':<15} {results['RAG']['time']:<12.2f} {results['RAG']['docs']:<8} {'Baseline':<30}")

# CAG feature
cag_feature = 'Cache: ' + ('HIT ⚡' if results['CAG']['cache_hit'] else 'MISS')
print(f"{'CAG':<15} {results['CAG']['time']:<12.2f} {results['CAG']['docs']:<8} {cag_feature:<30}")

# CRAG feature
crag_conf = results['CRAG']['confidence']
crag_emoji = ' ✅' if crag_conf > 0.7 else ' ⚠️'
crag_feature = f"Confidence: {crag_conf:.2f}{crag_emoji}"
print(f"{'CRAG':<15} {results['CRAG']['time']:<12.2f} {results['CRAG']['docs']:<8} {crag_feature:<30}")

# Summary recommendation
print("=" * 80)
print("💡 RECOMMENDATION")
print("=" * 80)

fastest = min(results.items(), key=lambda x: x[1]['time'])
print(f"⚡ Fastest: {fastest[0]} ({fastest[1]['time']:.2f}s)")

if results['CAG']['cache_hit']:
    print("🎯 Best Choice: CAG (cache hit = instant response)")
elif results['CRAG']['confidence'] > 0.7:
    print("🎯 Best Choice: CRAG (high confidence + corrective capability)")
else:
    print("🎯 Best Choice: Standard RAG (reliable baseline)")

print("\n✅ Inference complete!")
print("=" * 80)


🎤 INTERACTIVE RAG INFERENCE

📝 Ask your question about Nawaloka Hospital...


🔍 Processing query: 'Which PrimeLands projects are close to highway entrances?'


1️⃣  Standard RAG
--------------------------------------------------------------------------------
✅ Completed in 6.62s
📄 Documents retrieved: 4

💬 Answer:
**Key Facts**
*   BANDARAGAMA - GREEN PARADISE is 10 minutes to the Gelanigama highway entrance [https://www.primelands.lk/land/BANDARAGAMA-GREEN-PARADISE/en].
*   PRIME KESHARA ADAWIYA is 10 minutes to the Nakalagamuwa Highway Entrance [https://www.primelands.lk/land/PRIME-KESHARA-ADAWIYA/en].
*   PRIME ZONE 5 is close to the proposed Veyangoda Highway Interchange [https://www.primelands.lk/land/PRIME-ZONE-5/en].

**Answer**
Prime Lands has several projects that are close to highway entrances. The BANDARAGAMA - GREEN PARADISE project is located 10 minutes from the Gelanigama highway entrance [https://www.primelands.lk/land/BANDARAGAMA-GREEN-PARADISE/en]. PRIME KESHARA ADAWIY

In [9]:
print("=" * 80)
print("CAG PERFORMANCE TEST")
print("=" * 80)

test_queries = [
    "How do I contact Nawaloka primelands",
    "What services does primeland provide?"
]

# First run: populate cache
print("\n1️⃣  FIRST RUN (Populating cache)...\n")
for query in test_queries:
    result = cag_service.generate(query, use_cache=True, verbose=False)
    print(f"   Query: {query[:50]}...")
    print(f"   ⏱️  Time: {result['generation_time']:.2f}s | Cache: {result['cache_hit']}")
    print()

# Second run: cache hits
print("\n2️⃣  SECOND RUN (Using cache)...\n")
for query in test_queries:
    result = cag_service.generate(query, use_cache=True, verbose=False)
    print(f"   Query: {query[:50]}...")
    print(f"   ⏱️  Time: {result['generation_time']:.2f}s | Cache: {result['cache_hit']}")
    if result['cache_hit']:
        print(f"   🚀 INSTANT response from cache!")
    print()

print("\n📊 Cache Statistics:")
stats = cache.stats()
print(f"   Total cached: {stats['total_cached']}")
print(f"   Cache size: {stats['cache_size_kb']:.2f} KB")

CAG PERFORMANCE TEST

1️⃣  FIRST RUN (Populating cache)...

   Query: How do I contact Nawaloka primelands...
   ⏱️  Time: 4.93s | Cache: False

   Query: What services does primeland provide?...
   ⏱️  Time: 4.99s | Cache: False


2️⃣  SECOND RUN (Using cache)...

   Query: How do I contact Nawaloka primelands...
   ⏱️  Time: 0.55s | Cache: True
   🚀 INSTANT response from cache!

   Query: What services does primeland provide?...
   ⏱️  Time: 0.37s | Cache: True
   🚀 INSTANT response from cache!


📊 Cache Statistics:
   Total cached: 24
   Cache size: 635.31 KB


In [10]:
print("=" * 80)
print("CORRECTIVE RAG (CRAG) TEST")
print("=" * 80)

test_cases = [
    {
        'query': "SKYE BLOSSOM",
        'label': "Vague query (should trigger correction)"
    },
    {
        'query': "What is the price and payment plan for SKYE BLOSSOM KOTTAWA?",
        'label': "Specific query (should be confident)"
    },{
        'query': "land",
        'label': "Very vague query (should trigger correction)"
    },
    {
        'query': "Which PrimeLands projects are near highway entrances and what are their prices?",
        'label': "Detailed multi-part query (may need correction)"
    }
]

for i, test in enumerate(test_cases, 1):
    print(f"\nTest {i}/{len(test_cases)}: {test['label']}")
    print("-" * 80)
    
    result = crag_service.generate(test['query'], confidence_threshold=0.6)
    
    print(f"\n✅ Result:")
    print(f"   Initial confidence: {result['confidence_initial']:.2f}")
    print(f"   Final confidence: {result['confidence_final']:.2f}")
    print(f"   Correction applied: {result['correction_applied']}")
    print(f"   Documents used: {result['docs_used']}")
    print(f"   Generation time: {result['generation_time']:.2f}s")
    print("\n" + "-" * 80)

CORRECTIVE RAG (CRAG) TEST

Test 1/4: Vague query (should trigger correction)
--------------------------------------------------------------------------------
🔍 Query: SKYE BLOSSOM
🎯 Confidence threshold: 0.6

1️⃣  Initial retrieval (k=4)...
   📊 Confidence: 0.96
   ✅ Confidence sufficient - proceeding with initial retrieval

3️⃣  Generating answer...

✅ Result:
   Initial confidence: 0.96
   Final confidence: 0.96
   Correction applied: False
   Documents used: 4
   Generation time: 3.11s

--------------------------------------------------------------------------------

Test 2/4: Specific query (should be confident)
--------------------------------------------------------------------------------
🔍 Query: What is the price and payment plan for SKYE BLOSSOM KOTTAWA?
🎯 Confidence threshold: 0.6

1️⃣  Initial retrieval (k=4)...
   📊 Confidence: 0.76
   ✅ Confidence sufficient - proceeding with initial retrieval

3️⃣  Generating answer...

✅ Result:
   Initial confidence: 0.76
   Final con

In [11]:
print("🔍 Checking if all services are initialized...\n")

services_ready = True

# Check RAGService
try:
    rag_service
    print("✅ RAGService: Ready")
except NameError:
    print("❌ RAGService: NOT initialized")
    services_ready = False

# Check CAGService
try:
    cag_service
    print("✅ CAGService: Ready")
except NameError:
    print("❌ CAGService: NOT initialized")
    services_ready = False

# Check CRAGService
try:
    crag_service
    print("✅ CRAGService: Ready")
except NameError:
    print("❌ CRAGService: NOT initialized")
    services_ready = False

# Check Qdrant Vector Store
try:
    info = qdrant_client.collection_info(collection_name="prime_lands")
    print(f"✅ Qdrant: Connected ({info['points_count']} vectors, {info['distance']})")
except NameError:
    print("❌ Qdrant: NOT initialized")
    services_ready = False
except Exception as e:
    print(f"⚠️  Qdrant: Initialized but error - {e}")

print("\n" + "=" * 80)
if services_ready:
    print("🎉 All services ready! You can run the remaining cells.")
else:
    print("⚠️  Some services are missing. Please run all cells above first.")
print("=" * 80)


🔍 Checking if all services are initialized...

✅ RAGService: Ready
✅ CAGService: Ready
✅ CRAGService: Ready
✅ Qdrant: Connected (2015 vectors, COSINE)

🎉 All services ready! You can run the remaining cells.


---
# 📊 Comprehensive Comparison: RAG vs CAG vs CRAG

In [13]:
# Comprehensive Comparison
import pandas as pd

print("=" * 80)
print("📊 RAG vs CAG vs CRAG COMPARISON")
print("=" * 80)

comparison_query = "Which PrimeLands projects are near highway entrances and what are their prices?"

# Test 1: Standard RAG
print(f"\n1️⃣  Standard RAG...")
rag_result = rag_service.generate(comparison_query)
print(f"   ⏱️  Time: {rag_result['generation_time']:.2f}s")

# Test 2: CAG (should use cache on second run)
print(f"\n2️⃣  Cache-Augmented Generation (CAG)...")
cag_result = cag_service.generate(comparison_query, use_cache=True, verbose=False)
print(f"   ⏱️  Time: {cag_result['generation_time']:.2f}s")
print(f"   💾 Cache hit: {cag_result['cache_hit']}")

# Test 3: CRAG
print(f"\n3️⃣  Corrective RAG (CRAG)...")
crag_result = crag_service.generate(comparison_query, confidence_threshold=0.6, verbose=False)
print(f"   ⏱️  Time: {crag_result['generation_time']:.2f}s")
print(f"   🔧 Correction: {crag_result['correction_applied']}")

# Create comparison table
comparison_data = [
    {
        'Technique': 'Standard RAG',
        'Latency (s)': f"{rag_result['generation_time']:.2f}",
        'Docs Retrieved': rag_result['num_docs'],
        'Cache Used': 'No',
        'Self-Correcting': 'No',
        'Best For': 'General queries'
    },
    {
        'Technique': 'CAG',
        'Latency (s)': f"{cag_result['generation_time']:.2f}",
        'Docs Retrieved': 'Cached' if cag_result['cache_hit'] else rag_result['num_docs'],
        'Cache Used': 'Yes' if cag_result['cache_hit'] else 'No',
        'Self-Correcting': 'No',
        'Best For': 'Frequent queries'
    },
    {
        'Technique': 'CRAG',
        'Latency (s)': f"{crag_result['generation_time']:.2f}",
        'Docs Retrieved': crag_result['docs_used'],
        'Cache Used': 'No',
        'Self-Correcting': 'Yes',
        'Best For': 'Complex/uncertain queries'
    }
]

df = pd.DataFrame(comparison_data)

print("\n" + "=" * 80)
print("COMPARISON TABLE")
print("=" * 80)
print(df.to_string(index=False))

print("\n" + "=" * 80)
print("KEY INSIGHTS")
print("=" * 80)
print("✅ RAG: Baseline - reliable for general queries")
print("⚡ CAG: Fastest when cache hits - ideal for FAQs")
print("🎯 CRAG: Most accurate - self-corrects weak evidence")
print("💡 HYBRID: Combine all three for production!")
print("\n" + "=" * 80)

📊 RAG vs CAG vs CRAG COMPARISON

1️⃣  Standard RAG...
   ⏱️  Time: 7.38s

2️⃣  Cache-Augmented Generation (CAG)...
   ⏱️  Time: 5.81s
   💾 Cache hit: False

3️⃣  Corrective RAG (CRAG)...
   ⏱️  Time: 4.23s
   🔧 Correction: True

COMPARISON TABLE
   Technique Latency (s)  Docs Retrieved Cache Used Self-Correcting                  Best For
Standard RAG        7.38               4         No              No           General queries
         CAG        5.81               4         No              No          Frequent queries
        CRAG        4.23               8         No             Yes Complex/uncertain queries

KEY INSIGHTS
✅ RAG: Baseline - reliable for general queries
⚡ CAG: Fastest when cache hits - ideal for FAQs
🎯 CRAG: Most accurate - self-corrects weak evidence
💡 HYBRID: Combine all three for production!

